In [ ]:
%pip install -q bert_score
%pip install -q ragas

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
questions = [
    "아진산업의 최초 설립일은 언제인가요?",
    "아진산업의 2025년 매출액은 얼마인가요?",
    "아진산업의 본사 위치는 어디인가요?",
]

ground_truths = [
    "아진산업은 1978년 5월 31일 창립하였습니다.",
    "아진산업은 2025년에 1조 88억 6783만원의 매출 실적을 달성했습니다.",
    "아진산업의 본사는 경상북도 경산시 진량읍에 있습니다."
]

In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("..\\docs\\knu.pdf")
documents = loader.load()

C:\Users\USER\AppData\Local\Temp\ipykernel_32288\3768569987.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model="gpt-5.4-nano")
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding
)
retriever = vectorstore.as_retriever()

In [9]:
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
    다음 문맥만 사용하여 질문에 한 문장으로 답하세요.
    문맥에서 알 수 없는 내용은 모른다고 답하세요.
    [문맥] {context}
    [질문] {question}
""")

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [12]:
contexts = []
answers = []
for question in questions:
    docs = retriever.invoke(question)
    contexts.append([doc.page_content for doc in docs])
    answer = rag_chain.invoke(question)
    answers.append(answer)

In [13]:
import pandas as pd
from bert_score import score

P, R, F1 = score(
    cands=answers,
    refs=ground_truths,
    lang="ko",
    verbose=True
)

df_bertscore = pd.DataFrame({
    "question": questions,
    "ground_truth": ground_truths,
    "answer": answers,
    "bertscore_precision": P.tolist(),
    "bertscore_recall": R.tolist(),
    "bertscore_f1": F1.tolist(),
})

df_bertscore

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.67 seconds, 1.79 sentences/sec


,question,ground_truth,answer,bertscore_precision,bertscore_recall,bertscore_f1
0,아진산업의 최초 설립일은 언제인가요?,아진산업은 1978년 5월 31일 창립하였습니다.,문맥에서 아진산업의 최초 설립일에 대한 정보가 없어 모른다고 답하겠습니다.,0.739745,0.808107,0.772416
1,아진산업의 2025년 매출액은 얼마인가요?,아진산업은 2025년에 1조 88억 6783만원의 매출 실적을 달성했습니다.,문맥에는 아진산업의 2025년 매출액 정보가 없으므로 모릅니다.,0.779742,0.732850,0.755569
2,아진산업의 본사 위치는 어디인가요?,아진산업의 본사는 경상북도 경산시 진량읍에 있습니다.,문맥에서 아진산업의 본사 위치는 알 수 없으므로 모른다고 답합니다.,0.750971,0.772866,0.761761


In [14]:
df_bertscore["bertscore_f1"].mean()

np.float64(0.7632489204406738)

In [15]:
from datasets import Dataset

ragas_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
}

dataset = Dataset.from_dict(ragas_data)
dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 3
})

In [16]:
dataset[0]

{'question': '아진산업의 최초 설립일은 언제인가요?',
 'answer': '문맥에서 아진산업의 최초 설립일에 대한 정보가 없어 모른다고 답하겠습니다.',
 'contexts': ['- 118 -\n구분계열학과 또는 전공명입학정원 및 학과 또는 전공설치 여부석사과정2026학년도2025학년도2024학년도행 정 대 학 원인문․사회계열공 공 관 리 전 공○○○정 책․도 시 전 공○○○지 방 자 치 전 공○○○법 무․안 전 전 공○○○입학정원606060경 영 대 학 원인문․사회계열경 영 학 전 공○○○글 로 벌M B A전 공○○○입학정원120120120보 건 대 학 원자연과학계열역 학 및 건 강 증 진 학 과○○○보 건 관 리 학 과○○○입학정원404040산 업 대 학 원공 학 계 열산 업 공 학 과회 로  및  시 스 템 전 공○○○제 어  및  계 측 공 학 전 공○○○정 보 통 신 공 학 전 공○○○반 도 체 공 학 전 공○○○컴 퓨 터 공 학 전 공○○○기 계 공 학 전 공○○○응 용 화 학 전 공X○○금 속 신 소 재 공 학 전 공○○○건 축 공 학 전 공○○○토 목 공 학 전 공○○○전 기 공 학 전 공○○○화 학 공 학 전 공X○○응 용 화 학 공 학 전 공○XX고 분 자 공 학 전 공○○○무 기 재 료 공 학 전 공○○○섬 유 시 스 템 공 학 전 공○○○환 경 공 학 전 공○○○기 술 정 책 전 공○○○입학정원858585스 마 트 농 생 명식품융합대학원자연과학계열농 업 자 원 학 전 공○○○식 품 산 업 공 학 전 공○○○농 업 공 학 전 공○○○농 촌 개 발 전 공○○○농 산 물 안 전 성 학 전 공○○○농 업 정 책 및 유 통 전 공○○○환 경 조 경 학 전 공○○○스 마 트 농 업 시 스 템 공 학 전 공○○○입학정원353538',
  '- 131 -\n￭ 학사과정\n※ 과학기술대학 산업기계공학과, IT대학 ICT융합학과는 3학년 편입과정으로 운영※ 전자공학부 모바일AI공학전공은 학·석사통합과정으로 운영 \n관리 대학계열학과명[협약기관

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

ragas_result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=llm,
    embeddings=embedding,
)
ragas_result

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.8333, 'answer_relevancy': 0.0000, 'context_precision': 0.0000, 'context_recall': 0.0000}

In [18]:
df_ragas = ragas_result.to_pandas()
df_ragas

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,아진산업의 최초 설립일은 언제인가요?,[- 118 -\n구분계열학과 또는 전공명입학정원 및 학과 또는 전공설치 여부석사과...,문맥에서 아진산업의 최초 설립일에 대한 정보가 없어 모른다고 답하겠습니다.,아진산업은 1978년 5월 31일 창립하였습니다.,NaN,NaN,NaN,NaN
1,아진산업의 2025년 매출액은 얼마인가요?,[- 45 -\n[별표 1]<삭제 2006.9.29>[별표 2] <삭제 2020. ...,문맥에는 아진산업의 2025년 매출액 정보가 없으므로 모릅니다.,아진산업은 2025년에 1조 88억 6783만원의 매출 실적을 달성했습니다.,NaN,NaN,NaN,NaN
2,아진산업의 본사 위치는 어디인가요?,[- 102 -\n20. 2013학년도 대학원 학생정원 조정내용<2013. 3. 1...,문맥에서 아진산업의 본사 위치는 알 수 없으므로 모른다고 답합니다.,아진산업의 본사는 경상북도 경산시 진량읍에 있습니다.,NaN,NaN,NaN,NaN


In [19]:
df_eval = df_ragas.copy()

df_eval["bertscore_precision"] = df_bertscore["bertscore_precision"]
df_eval["bertscore_recall"] = df_bertscore["bertscore_recall"]
df_eval["bertscore_f1"] = df_bertscore["bertscore_f1"]

df_eval[
    [
        "question",
        "ground_truth",
        "answer",
        "bertscore_f1",
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
]

KeyError: "['question', 'ground_truth', 'answer'] not in index"

In [ ]:
score_columns = [
    "bertscore_f1",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]

df_eval[score_columns].mean()

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [15]:
import sys
!{sys.executable} -m pip show bert-score
!{sys.executable} -m pip show ragas
!{sys.executable} -m pip show langchain-google-vertexai

Name: bert-score
Version: 0.3.13
Summary: PyTorch implementation of BERT score
Home-page: https://github.com/Tiiiger/bert_score
Author: Tianyi Zhang*, Varsha Kishore*, Felix Wu*, Kilian Q. Weinberger, and Yoav Artzi
Author-email: tzhang@asapp.com
License: MIT
Location: d:\_knukdt\knu-kdt-dev\.kdtenv\Lib\site-packages
Requires: matplotlib, numpy, packaging, pandas, requests, torch, tqdm, transformers
Required-by: 
Name: ragas
Version: 0.4.3
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/vibrantlabsai/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or e

In [ ]:
import sys

!{sys.executable} -m pip install bert-score ragas langchain-google-vertexai

In [ ]:
import os
os._exit(00)